In [ ]:
# still need to make some improvements like edge cases for better fool proof
import sys

import torch.nn.functional as F
project_root = "/Users/sanedara/Documents/Machine_learning_practise/recommendation_system"
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from src.data.loader import MovieLensLoader
import pandas as pd
import numpy as np


In [ ]:
# get the data
loader = MovieLensLoader()
movies = loader.load_movies()
ratings = loader.load_ratings()
movies.head()

,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Childrens,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [3]:

# extract movie features and store them in an array, idx_to_index -> movie id to index, index_to_idx -> index to movie id mapping.
cols = movies.columns[6:] # get the genre colums since we recommend based on genres
movies_feats = movies.loc[:, cols].to_numpy(dtype=np.int8)
idx_to_index = pd.Series(movies.index.values, index=movies['movie_id']).to_dict()
index_to_idx = {v: k for k, v in idx_to_index.items()}


In [ ]:
# convert user data in to vector matrix

def user_feat_ext(user_data):
    movie_ids = user_data.loc[:,['item_id']].values.reshape(-1)
    user_feats = []
    for movie_id in movie_ids:
        user_feats.append(movies_feats[idx_to_index[movie_id]])
    user_feats = np.array(user_feats)
    user_vector = np.mean(user_feats, axis=0)
    return user_vector

def get_top_rec_movies(user_feat, seen_movies):
    u, v = np.linalg.norm(user_feat), np.linalg.norm(movies_feats, axis=1)
    cos_sim = np.dot(user_feat, movies_feats.T) / (u * v + 1e-09)
    top_movies_index = np.argsort(cos_sim)[-100:][::-1]
    recommend_idx = []
    for i in top_movies_index:
        if index_to_idx[i] not in seen_movies:
            recommend_idx.append(index_to_idx[i])
        if len(recommend_idx) >= 12:
            break
    return recommend_idx

            
def recommend_movies_with_cosine_sim(user_id):
    # filter user id and get data where the rating is more than 3
    user_data = ratings[ratings['user_id'] == user_id]
    user_data = user_data[user_data['rating'] >= 3]
    seen_movies = set(user_data['item_id'])
    # print(f"seen movies {seen_movies}")
    total_ratings = len(user_data)
    movie_names = ['not enough ratings for recommendation'] #cold start problem, show poplular suggestions implemented in baseline
    # get some data before showing any recommendations with a interested movies page
    if total_ratings > 15:
        user_feat = user_feat_ext(user_data)
        top_ids = get_top_rec_movies(user_feat, seen_movies)
        movie_names = movies.set_index('movie_id').loc[top_ids]['title']
    return movie_names
movie_names = recommend_movies_with_cosine_sim(100)
print(movie_names)

(44, 18)
[[0 0 1 ... 0 0 0]
 [1 1 0 ... 1 0 0]
 [0 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]] (1682, 18)
0.812162199327529
